In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/2.1 silver_transformation_framework"

In [0]:
# ---------------------------------------------------------
# 0. Silver Transformation Framework Parameters
# ---------------------------------------------------------

schema = StructType([
    StructField("Type", StringType(), True),
    StructField("Subtype", StringType(), True),
    StructField("Account", StringType(), True),
    StructField("SubAccount", StringType(), True),
    StructField("ValueType", StringType(), True),
    StructField("Account_key", IntegerType(), True),

])

bronze_table = "fake_general_ledger.default.bronze_statement_of_cash_flow"

silver_table = "fake_general_ledger.default.silver_statement_of_cash_flow"  

merge_condition = """target.type = source.type 
                  AND target.subtype = source.subtype
                  AND target.account = source.account
                  AND target.subaccount = source.subaccount
                  AND target.valuetype = source.valuetype
                  AND target.account_key = source.account_key"""

required_columns = [
    "type",
    "subtype",
    "account",
    "subaccount",
    "valuetype",
    "account_key",
]

primary_key_columns = [
    "type",
    "subtype",
    "account",
    "subaccount",
    "valuetype",
    "account_key",
]

order_column = "type"



In [0]:
# ---------------------------------------------------------
# 1. Silver Transformation Driver Code
# ---------------------------------------------------------

bronze_df = spark.table(
    bronze_table
)

bronze_df = standardize_column_names(
    bronze_df
)

silver_df = apply_silver_schema(
    bronze_df,
    schema
)

silver_df, invalid_df = validate_required_columns(
    silver_df,
    required_columns
)

silver_df = deduplicate_records(
    silver_df,
    primary_key_columns,
    order_column
)

silver_df = apply_silver_metadata(
    silver_df
)

silver_df = rename_bronze_metadata(
    silver_df
)

silver_df = drop_excess_bronze_metadata(
    silver_df
)

merge_to_silver(
    spark,
    silver_df,
    silver_table,
    merge_condition
)